In [26]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [9]:
train_data = pd.read_csv("Kaggle_Data/train.csv")
# train_data["filename"] = train_data["path"].str.split("/").str[-1]
# train_data.drop(columns=["path"], inplace=True)
train_data.head()


,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie


In [4]:
with open("Kaggle_Data/sign_to_prediction_index_map.json", 'r') as f:
    sign_to_prediction_index_map = json.load(f)

In [ ]:
landmark_data = pd.read_parquet("Kaggle_Data/train_landmark_files/16069/100015657.parquet")
landmark_data.head()

,frame,row_id,type,landmark_index,x,y,z
0,103,103-face-0,face,0,0.437886,0.437599,-0.051134
1,103,103-face-1,face,1,0.443258,0.392901,-0.067054
2,103,103-face-2,face,2,0.443997,0.409998,-0.042990
3,103,103-face-3,face,3,0.435256,0.362771,-0.039492
4,103,103-face-4,face,4,0.443780,0.381762,-0.068013


In [21]:
one_frame = landmark_data[landmark_data["frame"] == 103]
for landmark_type, type_data in one_frame.groupby("type"):
    print(landmark_type)
    print(type_data.shape)
    # print(type_data.head())

face
(468, 7)
left_hand
(21, 7)
pose
(33, 7)
right_hand
(21, 7)


In [47]:
landmark_consolidated = []

for frame, frame_df in landmark_data.groupby("frame"):
    frame_data = []
    
    for landmark_type, type_data in frame_df.groupby("type"):
        
        if landmark_type == "face":
            continue

        type_data = type_data.fillna(0)
        coordinates = type_data.reset_index()[['x', 'y']].values.tolist()
        frame_data.extend(coordinates)

    landmark_consolidated.append(frame_data)    


In [48]:
landmark_consolidated = np.asarray(landmark_consolidated).reshape(1, 105, 150)
landmark_consolidated.shape

(1, 105, 150)

In [41]:
parquet_files_path = Path(
    r"Kaggle_Data/train_landmark_files"
)

for signer_folder in parquet_files_path.iterdir():
    if signer_folder.name == '.DS_Store':
        continue

    for parquet_file in signer_folder.iterdir():

        signing_word = train_data[
            (train_data['participant_id'] == int(parquet_file.parent.name)) & 
            (train_data['sequence_id'] == int(parquet_file.stem))
        ]['sign'].iloc[0]

        signing_word_id = sign_to_prediction_index_map[signing_word]

In [64]:
prediction = np.zeros(len(sign_to_prediction_index_map))
prediction[signing_word_id] = 1
prediction = prediction.reshape(1, -1)

In [50]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [59]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return self.softmax(out)

In [55]:
input_size = landmark_consolidated.shape[2]
hidden_size = 64
num_layers = 2
num_classes = len(sign_to_prediction_index_map)

model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes)
model.to(device)

model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [68]:
X_tensor = torch.tensor(landmark_consolidated, dtype=torch.float32).to(device)
Y_tensor = torch.tensor(prediction, dtype=torch.float32).to(device)

In [70]:
num_epochs = 10

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, Y_tensor)
    loss.backward()
    optimizer.step()
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')



Epoch [1/10], Loss: 5.723202228546143
Epoch [2/10], Loss: 5.6187520027160645
Epoch [3/10], Loss: 5.518866539001465
Epoch [4/10], Loss: 5.417169570922852
Epoch [5/10], Loss: 5.308719158172607
Epoch [6/10], Loss: 5.189612865447998
Epoch [7/10], Loss: 5.056641101837158
Epoch [8/10], Loss: 4.907191276550293
Epoch [9/10], Loss: 4.739643573760986
Epoch [10/10], Loss: 4.553947448730469
